In [ ]:
!pip install -q timm einops peft wandb deepspeed accelerate bitsandbytes decord tensorboardX gdown

In [ ]:
!pip install -U datasets
!pip install transformers==4.47.0
# !pip install flash_attn==2.7.2.post1

In [ ]:
!git clone https://github.com/5CD-AI/Vintern.git
%cd Vintern

In [5]:
import os
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import matplotlib.pyplot as plt

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    #check image_file is a path or image
    if isinstance(image_file, str):
        image = Image.open(image_file).convert('RGB')
    else:
        image = image_file
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

In [ ]:
## load pretrained model
model_name = "5CD-AI/Vintern-1B-v3_5"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    use_flash_attn=False,
).eval().cuda()

In [7]:
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
# generation_config = dict(max_new_tokens= 1024, do_sample=False, num_beams = 3, repetition_penalty=2.0)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
generation_config = dict(
    max_new_tokens=2048,
    do_sample=True,        # Bật lấy mẫu thay vì tìm kiếm cứng nhắc
    temperature=0.1,       # Nhiệt độ cực thấp để mô hình không bịa chữ (rất quan trọng)
    top_p=0.9,
    repetition_penalty=1.05
)

In [8]:
import json, re

ALLOWED_KEYS = {
    "dosage_form", "packaging", "uses",
    "contraindications", "side_effects",
    "dosage_administration", "storage_condition", "warning"
}

In [9]:
def parse_result(response):
    try:
        # Làm sạch chuỗi trước khi parse
        clean = response.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1]
        if "```" in clean:
            clean = clean.split("```")[0]

        raw = json.loads(clean.strip())
        result = {}
        for key in ALLOWED_KEYS:
            value = raw.get(key, None)
            if not value or str(value).strip().lower() == "null" or str(value).strip() == "":
                result[key] = None
            else:
                result[key] = value
        return result
    except Exception as e:
        print(f"Lỗi parse JSON: {e}")
        return {key: None for key in ALLOWED_KEYS}

In [ ]:
test_image = '/content/hapacol325.webp'

pixel_values = load_image(test_image, max_num=12).to(torch.bfloat16).cuda()

medicine_prompt = '''<image>
Hãy đọc toàn bộ thông tin trong tờ hướng dẫn sử dụng thuốc.
Sau khi đọc xong, BẮT BUỘC trích xuất các thông tin vừa đọc được vào định dạng JSON sau (nếu không có thông tin thì để null):
{
  "dosage_form": "Dạng bào chế...",
  "packaging": "Quy cách đóng gói...",
  "uses": "Công dụng hoặc Chỉ định...",
  "contraindications": "Chống chỉ định...",
  "side_effects": "Tác dụng phụ...",
  "dosage_administration": "Liều dùng và cách dùng...",
  "storage_condition": "Điều kiện bảo quản...",
  "warning": "Cảnh báo, thận trọng hoặc lưu ý..."
}
'''

# medicine_prompt = '''<image>
# Hãy đóng vai một chuyên gia y tế, đọc toàn bộ nội dung tờ hướng dẫn sử dụng thuốc trong ảnh.
# Nhiệm vụ của bạn là trích xuất thông tin và BẮT BUỘC TRẢ VỀ ĐỊNH DẠNG JSON hợp lệ. Không thêm bất kỳ văn bản, lời chào, hay giải thích nào bên ngoài khối JSON.

# Sử dụng chính xác các key tiếng Anh dưới đây, điền nội dung tương ứng bằng tiếng Việt. Nếu trong ảnh không đề cập đến thông tin đó, hãy để giá trị là null:
# {
#   "dosage_form": "Dạng bào chế...",
#   "packaging": "Quy cách đóng gói...",
#   "uses": "Công dụng, chỉ định...",
#   "contraindications": "Chống chỉ định...",
#   "side_effects": "Tác dụng phụ...",
#   "dosage_administration": "Liều dùng và cách dùng...",
#   "storage_condition": "Điều kiện bảo quản...",
#   "warning": "Cảnh báo, thận trọng hoặc lưu ý, cảnh báo và thận trọng..."
# }
# '''

response = model.chat(tokenizer, pixel_values, medicine_prompt, generation_config)
print(response)
del pixel_values
result = parse_result(response)

print(json.dumps(result, ensure_ascii=False, indent=2))

In [10]:
!pip install -q flask flask-cors pyngrok

In [11]:
# Setup Ngrok Token
from google.colab import userdata
from flask import Flask, jsonify, request
from flask_cors import CORS
from pyngrok import ngrok

authtoken = userdata.get("ngrok_token")
ngrok.set_auth_token(authtoken)

In [ ]:
from PIL import Image

app = Flask(__name__)
CORS(app)

medicine_prompt = '''<image>
Hãy đọc toàn bộ thông tin trong tờ hướng dẫn sử dụng thuốc.
Sau khi đọc xong, BẮT BUỘC trích xuất các thông tin vừa đọc được vào định dạng JSON sau (nếu không có thông tin thì để null):
{
  "dosage_form": "Dạng bào chế...",
  "packaging": "Quy cách đóng gói...",
  "uses": "Công dụng hoặc Chỉ định...",
  "contraindications": "Chống chỉ định...",
  "side_effects": "Tác dụng phụ...",
  "dosage_administration": "Liều dùng và cách dùng...",
  "storage_condition": "Điều kiện bảo quản...",
  "warning": "Cảnh báo, thận trọng hoặc lưu ý..."
}
'''

@app.route('/ocr/medicine', methods=['POST'])
def ocr_medicine():
    if 'image' not in request.files:
        return jsonify({"error": "Không tìm thấy ảnh"}), 400

    file = request.files['image']
    image = Image.open(file.stream).convert('RGB')

    transform = build_transform(input_size=448)
    images = dynamic_preprocess(image, image_size=448, use_thumbnail=True, max_num=6)
    pixel_values = torch.stack([transform(img) for img in images]).to(torch.bfloat16).cuda()

    response = model.chat(tokenizer, pixel_values, medicine_prompt, generation_config)
    del pixel_values

    result = parse_result(response)
    return jsonify(result)

if __name__ == '__main__':
    ngrok_url = ngrok.connect(5555)
    print("API URL:", ngrok_url)
    app.run(port=5555)